# Acesso Territorial a Oportunidades

Este notebook gera **mapas de calor** que mostram onde se concentram três tipos de oportunidades: (i) empregos formais, (ii) infraestrutura de saúde pública e (iii) oferta de educação básica na rede pública. Pressupõe-se que o usuário disponha dos arquivos resultantes desses fluxos, o que pode ser feito com outros scripts constantes deste Kit. O código carrega o conjunto pertinente ao domínio escolhido (emprego, saúde ou educação), aplica filtros temáticos — nível de complexidade hospitalar ou etapa de ensino, quando solicitados — e gera mapas de acessibilidade por transporte público ou por caminhada em intervalosde tempo pré-definidos — futuras versões irão conter mecanismos para estimar mais adequadamente o acesso por transporte individual.

Além de mapear o acesso a essas oportunidades, o notebook também **relaciona cada superfície de acessibilidade à distribuição da população local por sexo, raça, renda e faixa etária**. Essa comparação, permite avaliar desigualdades socioespaciais e orientar políticas de mobilidade e inclusão.

O resultado facilita avaliações de equidade, diagnósticos de mobilidade e cálculos de acessibilidade focados nos serviços públicos essenciais.

> Em síntese, há fluxo único que, a partir de arquivos já processados, possibilite:  
> • **Selecionar o domínio** de análise – emprego, saúde ou educação;  
> • **Aplicar filtros temáticos** (complexidade hospitalar ou etapa escolar) quando pertinentes;  
> • **Modele a rede de transporte** a partir da rede viária e (opcionalmente) de um feed GTFS informado pelo usuário;  
> • **Calcular mapas de calor** do acesso às oportunidades públicas escolhidas;  
> • **Comparar essas densidades com camadas demográficas** (população total e recortes por sexo, raça, renda e idade) para medir quão equitativamente os serviços estão distribuídos.



## PANORAMA

- **Arquivos de entrada**  
  - `EMP_FILE` Empregos formais.  
  - `HEALTH_FILE` Estabelecimentos de saúde pública com leitos, consultórios e profissionais.  
  - `EDUC_FILE` Escolas públicas com matrículas e quadro de funcionários por etapa de ensino.  
  - `POP_FILE` Dados demográficos com população e atributos socioeconômicos.  
  - `GTFS_FILE` Feed GTFS (ZIP) representando a oferta de transporte coletivo no mesmo período de análise.  

- **Fluxo de processamento**  
  1. **Leitura** dos Parquet e do GTFS.  
  2. **Filtragem temática** (complexidade ou etapa).  
  3. **Validação de CRS** — todos os pontos já são geocodificados; o GTFS é convertido ao mesmo sistema.  
  5. **Construção da rede** — matriz tempo-custo porta-a-porta via r5py.  
  6. **Intersecção com população** — contagem de oportunidades acessíveis em 30 min e 60 min por célula e grupo demográfico.  
  7. **Visualização** — mapas interativos (densidade e acessibilidade), histogramas e relatórios em `OUTPUT_DIR`.


## LIMITAÇÕES

1. **Acessibilidade potencial** 


## Tempo de Caminhada a Equipamentos Públicos: Saúde e Educação

**AUTORIA:** [REDE MOB](https://www.redemob.com.br/), a partir de [um repositório no GitHub](https://github.com/eemilhaa/walkability-analysis).

Este script usa o r5py para calcular o tempo necessário para chegar a escolas ou unidades de saúde, a pé ou por transporte público. A rede de transporte é construída a partir da rede viária contida no [OpenStreetMap](https://wiki.openstreetmap.org/), a qual é combinada com uma rede de transporte público, se fornecido um [feed GTFS](https://en.wikipedia.org/wiki/GTFS). Os locais de interesse — aqui, escolas ou estabelecimentos de saúde de saúde — vêm da [Base dos Dados](https://basedosdados.org/), garantindo fontes confiáveis e atualizadas. A partir disso, é possível escolher entre duas formas de medir acessibilidade: o tempo até a oportunidade mais próxima ou a quantidade de oportundades que podem ser alcançados dentro de um certo limite de tempo.

Com ou sem transporte público, o foco é o mesmo: entender quem pode acessar quais serviços, e em quanto tempo.

**PANORAMA:**
- Usa r5py para cálculo de tempos de viagem multimodal (caminhada e, opcionalmente, transporte público).

- POIs (escolas e unidades de saúde) são extraídos da Base dos Dados.

- Utilização de malha haxagonal [H3](https://www-uber-com.translate.goog/en-BR/blog/h3/?_x_tr_sl=en&_x_tr_tl=pt&_x_tr_hl=pt&_x_tr_pto=tc), garantindo cobertura espacial regular.

- Suporte a análise com ou sem GTFS para considerar transporte público.

- Métricas de acessibilidade: (i) tempo até a oportunidade mais próxima ou (ii) número de POIs acessíveis dentro de um limite de tempo.

**MAIS INFORMAÇÕES:**
- [Layout da Plataforma]
- [Sumário dos Dados Disponíveis]
- *Lorem ipsum: Conteúdo do MOB de interesse, técnico ou de divulgação*

**LINKS DE INTERESSE:**
- links para materiais técnicos e acadêmicos gerais, externos, de referência a respeito do conteúdo abordado


# Instruções

## A. Requisitos

### 1. Chave do Google Cloud
- Forneça uma chave referente a um [projeto no Google Cloud]((https://basedosdados.org/docs/access_data_bq)).

### 2. Código IBGE do Município
- Informe o [código IBGE](https://www.ibge.gov.br/explica/codigos-dos-municipios.php) do município que deseja analisar.
- Para incluir municípios vizinhos, forneça uma **lista** com os códigos IBGE.

### 3. Tipo de Oportunidade
Escolha entre `saude` ou `educacao`:

#### Se `educacao`:
- Use `nivel_ensino` para selecionar entre:
  - `infantil`, `fundamental`, `medio`, `eja`, `profissional`
  - Ou uma lista combinando esses níveis.
- Use `categoria_escola` para filtrar por tipo de escola:
  - `privada`, `publica` ou `todas` (padrão).

#### Se `saude`:
- Ajuste as variáveis:
  - `vinculo_sus`: `'todos'`, `'publico'` ou `'privado'`.
  - `nivel_complexidade`: `'todos'`, `'baixa'`, `'media'`, `'alta'`.
  - `apenas_hospital`: `True` (apenas hospitais) ou `False` (padrão).

### 4. Transporte
#### Caminhada:
- Deixe a variável `gtfs` como `None`.

#### Transporte Público:
- Forneça um **arquivo GTFS** na variável `gtfs`.
- Defina uma **hora de partida** no formato `HH:MM`.

### 5. Configurações Padrão
- `velocidade_caminhada`: `4.5` km/h (1,25 m/s), baseada na [velocidade média de adultos saudáveis](https://doi.org/10.1007/s40279-020-01351-3).
- `tempo_maximo_viagem**`: `30` minutos. Locais mais distantes são considerados fora de alcance.
- `medida_acessibilidade`: Escolha entre:
  - `tempo_minimo`: Tempo até o ponto de interesse (POI) mais próximo.
  - `cumulativa`: Medida cumulativa de acesso (quantidade de POIs acessíveis dentro do tempo máximo).

## B. Exemplos de Configuração

1. **Análise de Educação**:

        tipo_oportunidade = 'educacao'
        nivel_ensino = 'fundamental'
        categoria_escola = 'publica'

2. **Análise de Saúde**:

        tipo_oportunidade = 'saude'
        vinculo_sus = 'publico'
        nivel_complexidade = 'media'
        apenas_hospital = True

3. **Transporte Público**:

        gtfs = 'caminho/para/arquivo.gtfs.zip'
        hora_partida = '07:30'


# Parâmetros Definidos Pelo Usuário

In [ ]:
# Parâmetros de Configuração

ibge_ids = [3303302, 3304904, 3301900]  # Niterói, São Gonçalo e Itaboraí
tipo_oportunidade = 'educacao'  # 'educacao' ou 'saude'

# Configurações para 'educacao'
nivel_ensino = ['medio']  # 'infantil', 'fundamental', 'medio', 'eja', 'profissional', ou lista
categoria_escola = 'publica'  # 'publica', 'privada', ou 'todas'

# Configurações para 'saude'
vinculo_sus = 'publico'  # 'todos', 'publico', ou 'privado'
nivel_complexidade = 'todos'  # 'todos', 'basica', 'media', ou 'alta'
apenas_hospital = True  # True para apenas hospitais

# Transporte
gtfs = None #"../database/4. Dados Operacionais do Transporte/gtfs_rio-de-janeiro.zip"  # Caminho para o arquivo GTFS ou None para caminhada
hora_partida = '07:30'  # Formato 'HH:MM' (necessário se GTFS for fornecido)

# Medidade de acessibilidade
medida_acessibilidade = 'tempo_minimo'  # 'tempo_minimo' ou 'cumulativa'

# Configurações gerais
velocidade_caminhada = 4.5  # km/h
tempo_maximo_viagem = 120  # minutos

In [ ]:
# Chave do serviço de nuvem do Google
from getpass import getpass
gcloud_id = getpass('Chave do serviço de nuvem do Google:')

# Ajustar Acima

# Backend

Processamento interno do código. A princípio, o usuário não precisa se preocupar com esta parte, mas aqueles com conhecimento mais avançado de programação podem fazer ajustes de acordo com as próprias necessidades específicas.

In [10]:
import datetime as dt
import difflib
import json
import subprocess
import tempfile
from collections.abc import Iterable
from pathlib import Path
from typing import Optional, Tuple, Union

# --- Third-party ---
import basedosdados as bd
import contextily as ctx
import geobr
import geopandas as gpd
import h3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import partridge as ptg
import r5py
import seaborn as sns
from matplotlib_map_utils.core.north_arrow import north_arrow
from matplotlib_scalebar.scalebar import ScaleBar
from mpl_toolkits.axes_grid1 import make_axes_locatable
from shapely.geometry import MultiPolygon, Polygon, mapping
from shapely.geometry.base import BaseGeometry
from statsmodels.stats.weightstats import DescrStatsW
from tobler.util import h3fy

import shutil, tempfile
from pathlib import Path
from pathlib import Path
import shutil, tempfile
import subprocess, shlex

%matplotlib inline
%config InlineBackend.figure_format = 'retina'


In [11]:
MUNIS          = [3304557, 3303302, 3304904, 3301900]


OUTPUT_DIR     = Path("./database/1. Socioeconômicos")

OUT_PARQUET    = (
    OUTPUT_DIR
    / f"sociodemografia_hex_r9_{'-'.join(str(m) for m in MUNIS)}.parquet"
    )

hexes = (
    gpd
    .read_parquet(OUT_PARQUET)
    .pipe(
        lambda x: x.loc[x.year==2022]
        )
    .reset_index()
    )

In [12]:
input_dir = "../database/4. Dados Operacionais do Transporte/GTFS"
gtfs_inpath = [
    f"{input_dir}/Ônibus RJ/gtfs_rio-de-janeiro.zip",
    f"{input_dir}/Barcas/gtfs_barcas.zip",
    f"{input_dir}/Metrô/gtfs_metro.zip",
    f"{input_dir}/Supervia/gtfs_supervia.zip",
    f"{input_dir}/VLT/gtfs_vlt.zip",
    f"{input_dir}/Leste/gtfs_itaborai.zip",
    f"{input_dir}/Leste/gtfs_niteroi.zip",
    f"{input_dir}/Leste/gtfs_sg.zip",
    f"{input_dir}/gtfs_intermunicipal.zip",
    ]

pbf_path = "../database/rmrj.osm.pbf"

In [13]:
# --- Region & time helpers ---------------------------------------------------
def get_departure_datetime(gtfs_path: Union[str, Path], time_str: str) -> dt.datetime:
    """
    Combine the GTFS busiest service date with a 'HH:MM' clock time.
    Requires a `ptg.read_busiest_date(gtfs_path)` function in scope.
    """
    if isinstance(gtfs_path, list):
        gtfs_path = gtfs_path[0]

    busiest_date, _ = ptg.read_busiest_date(gtfs_path)
    
    try:
        hour, minute = map(int, time_str.split(":"))
    except Exception as exc:
        raise ValueError(f"Hora inválida: '{time_str}'. Formato esperado: 'HH:MM'") from exc
    return dt.datetime.combine(busiest_date, dt.time(hour, minute))


# --- R5 transport network (cached) -----------------------------------------
#@lru_cache(maxsize=4)
def _cache_transport_network(
    pbf_path: str,
    gtfs_paths: tuple[str, ...]
) -> r5py.TransportNetwork:
    """
    Build and cache an R5 TransportNetwork. Cache key is (pbf_path, gtfs_paths).
    """
    kwargs = {"osm_pbf": pbf_path}
    if gtfs_paths:
        kwargs["gtfs"] = list(np.atleast_1d(gtfs_paths))
    return r5py.TransportNetwork(**kwargs)

# --- TravelTimeMatrix argument builder ---------------------------------------

def ttime_matrix_args(
    *,
    transport_network: r5py.TransportNetwork,
    max_travel_time: int,
    origins: gpd.GeoDataFrame,
    aperture: int,
    gtfs_path: Optional[Union[str, Path]],
    departure_time_str: Optional[str],
    walk_speed: float,  # km/h
    destinations: gpd.GeoDataFrame = None,
    overrides: Optional[dict] = None,
) -> dict:
    """
    Assemble keyword arguments for r5py.TravelTimeMatrix with clear defaults.
    """
    args = {
        "transport_network": transport_network,
        "origins": origins,
        "destinations": destinations,
        "max_time": dt.timedelta(minutes=max_travel_time),
        "transport_modes": [r5py.TransportMode.WALK],
        "speed_walking": float(walk_speed),
        "snap_to_network": float(
            np.ceil(h3.average_hexagon_edge_length(aperture, unit="m"))
            ),
    }

    # Transit branch only if GTFS + departure are provided
    if gtfs_path and departure_time_str:
        args["departure"] = get_departure_datetime(gtfs_path, departure_time_str)
        args["departure_time_window"] = dt.timedelta(hours=2)
        args["max_public_transport_rides"] = 2
        args["transport_modes"] = [r5py.TransportMode.TRANSIT, r5py.TransportMode.WALK]
        args["max_time"] = dt.timedelta(minutes=max_travel_time)

    if overrides:
        args.update(overrides)

    return args


def _get_origins(study_area):
    origins = study_area.to_crs(4326)
    origins["geometry"] = origins.geometry.centroid
    return origins.rename(columns={'hex_id': 'id'})

# --- Main pipeline: polygon clip → cached network → TTM ----------------------

def compute_travel_time_matrix(
    study_area: gpd.GeoDataFrame,
    *,
    origins: gpd.GeoDataFrame = None,
    destinations: gpd.GeoDataFrame = None,
    pbf_path: Union[str, Path] = None,
    gtfs_path: Optional[Union[str, Path, Iterable[Union[str, Path]]]] = None,
    aperture: int = 9,
    departure_time_str: str = "07:00",
    max_travel_time: int = 60,
    walk_speed: float = 4.5,  # km/h
    overrides: Optional[dict] = None,
) -> gpd.GeoDataFrame:
    """
    Compute a Travel Time Matrix (R5) using a **polygon-only** OSM clip:
    - clip PBF by the convex hull of *study_area*,
    - reuse a cached R5 network,
    - use centroids of *study_area* as origins.

    Returns
    -------
    GeoDataFrame with travel times; rows correspond to input centroids.
    """
    network = _cache_transport_network(
        pbf_path=pbf_path, gtfs_paths=gtfs_path
        )

    # 4) Compute TTM
    print("🚌 Network ready! Computing travel times…")
    args = ttime_matrix_args(
        transport_network=network,
        max_travel_time=max_travel_time,
        origins=_get_origins(study_area) if origins is None else origins,
        destinations=origins if destinations is None else destinations,
        aperture=aperture,
        gtfs_path=gtfs_path,
        departure_time_str=departure_time_str,
        walk_speed=walk_speed,
        overrides=overrides,
    )
    return r5py.TravelTimeMatrix(**args).dropna()


In [14]:
destinations = _get_origins(hexes).to_crs(31983)

In [15]:
def load_municipal_boundaries_rmrj(year: int = 2022,
                                   simplified: bool = True) -> gpd.GeoDataFrame:
    """
    Lê limites municipais (geobr) de Niterói, São Gonçalo e Itaboraí
    e retorna um único GeoDataFrame concatenado.
    """
    # Códigos IBGE
    codes = {
        "Niteroi": 3303302,
        "Sao Goncalo": 3304904,
        "Itaborai": 3301900,
    }
    parts = []
    for label, code in codes.items():
        gdf = geobr.read_municipality(code_muni=code,
                                      year=year,
                                      simplified=simplified)
        # Mantém o nome original de geobr e adiciona um rótulo simples sem acentos
        gdf["muni_label"] = label
        parts.append(gdf)

    out = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=parts[0].crs)
    return out.to_crs(31983)

In [16]:
origins = destinations.sjoin(
    load_municipal_boundaries_rmrj(),
    predicate="within"
).drop_duplicates('id')

In [17]:
ttime = compute_travel_time_matrix(
    study_area=hexes,
    pbf_path=pbf_path,
    gtfs_path=gtfs_inpath,
    origins=origins,
    destinations=destinations,
    aperture=9,
    #departure_time_str=hora_partida,
    max_travel_time=180,
    departure_time_str="05:00",
    #walk_speed=velocidade_caminhada,
    overrides={
        "departure_time_window": dt.timedelta(hours=3),
        },
)

🚌 Network ready! Computing travel times…


In [18]:
ttime.to_parquet(
    '../outputs/data/ttimes_2transfers_180min.parquet'
    )

In [19]:
ttime = compute_travel_time_matrix(
    study_area=hexes,
    pbf_path=pbf_path,
    gtfs_path=gtfs_inpath,
    aperture=9,
    origins=origins,
    destinations=destinations,
    max_travel_time=180,
    departure_time_str="05:00",
    #walk_speed=velocidade_caminhada,
    overrides={
        "departure_time_window": dt.timedelta(hours=3),
        'max_public_transport_rides': 3,
        },
)

ttime.to_parquet(
    '../outputs/data/ttimes_3transfers_180min.parquet'
    )

🚌 Network ready! Computing travel times…


In [ ]:
def merge_opportunities(travel_matrix, land_uses, opportunity, active):
    id_col = "to_id" if active else "from_id"
    return travel_matrix.merge(
        land_uses[["hex_id", opportunity]],
        left_on=id_col,
        right_on="hex_id",
        how="left"
    )


def cumulative_accessibility_score(data, opportunity, threshold, group_cols):
    accessible = data[data["travel_time"] <= threshold]
    return (
        accessible
        .groupby(group_cols)[opportunity]
        .sum()
        .reset_index(name="access_raw")
    )


def min_cost_for_threshold(data, opportunity, cost_col, threshold, group_cols):
    eligible = data[data[opportunity] > 0]
    eligible = eligible.sort_values(group_cols + [cost_col])
    eligible["cumsum"] = eligible.groupby(group_cols)[opportunity].cumsum()

    if threshold == 1:
        return (
            eligible.groupby(group_cols)[cost_col]
            .min()
            .reset_index(name="min_cost")
        )

    qualified = eligible[eligible["cumsum"] >= threshold]
    return (
        qualified.groupby(group_cols)[cost_col]
        .min()
        .reset_index(name="min_cost")
    )


def cost_to_closest_score(data, opportunity, cost_col, thresholds, group_cols):
    results = []
    for value in thresholds:
        cost = min_cost_for_threshold(
            data, opportunity, cost_col, value, group_cols
        )
        cost["n"] = value
        results.append(cost)

    return pd.concat(results, ignore_index=True)


def generate_missing_combinations(travel_matrix, id_col, group_by, thresholds):
    ids = travel_matrix[id_col].unique()
    group_vals = {g: travel_matrix[g].unique() for g in group_by}

    parts = [pd.Series(ids, name=id_col)]
    for g in group_by:
        parts.append(pd.Series(group_vals[g], name=g))
    parts.append(pd.Series(thresholds, name="n"))

    return pd.MultiIndex.from_product(parts, names=[id_col] + group_by + ["n"])


def fill_missing_combinations(df, travel_matrix, id_col, group_by, thresholds):
    index = generate_missing_combinations(travel_matrix, id_col, group_by, thresholds)
    df = df.set_index([id_col] + group_by + ["n"])
    df = df.reindex(index).reset_index()
    return df


def run_cumulative_method(data, land_uses, opportunity, threshold, id_col, group_cols):
    scores = cumulative_accessibility_score(
        data, opportunity, threshold, group_cols
    )
    total = land_uses[opportunity].sum()
    result = land_uses.merge(
        scores,
        left_on="hex_id",
        right_on=id_col,
        how="left"
    ).fillna({"access_raw": 0})
    result["access_norm"] = result["access_raw"] / total
    return result


def run_cost_to_closest_method(
    data, travel_matrix, opportunity, cost_col,
    thresholds, id_col, group_by, fill_missing_ids
):
    group_cols = [id_col] + group_by
    scores = cost_to_closest_score(
        data, opportunity, cost_col, thresholds, group_cols
    )

    if fill_missing_ids:
        scores = fill_missing_combinations(
            scores, travel_matrix, id_col, group_by, thresholds
        )

    scores = scores.rename(columns={id_col: "id"})
    cols = ["id"] + group_by + (["n"] if len(thresholds) > 1 else []) + ["min_cost"]
    return scores[cols]


def compute_accessibility(
    travel_matrix,
    land_uses,
    opportunity,
    method="cumulative",
    travel_cost_col="travel_time",
    threshold=60,
    n=None,
    group_by=None,
    active=True,
    fill_missing_ids=True
):
    """
    Computes accessibility metrics based on a travel time matrix and land use data.

    Supports two methods:
    - 'cumulative': sums accessible opportunities within a travel time threshold.
    - 'cost_to_closest': returns the minimum travel cost to reach at least n 
      opportunities.

    Parameters:
        travel_matrix (pd.DataFrame):
            Origin-destination matrix with travel times and at least 'from_id',
            'to_id', and travel cost columns. 'from_id' and 'to_id' should refer
            to the sameID column in `land_uses`.
        land_uses (gpd.GeoDataFrame):
            Regular (hexagonal) grid containing the `opportunity` column.
        opportunity (str):
            Name of the column in `land_uses` representing the opportunity variable
            (e.g., 'schools', 'jobs').
        travel_cost_col (str):
            Name of the column in `travel_matrix` representing travel time or cost.
        method (str):
            Accessibility method to compute: "cumulative" or "cost_to_closest".
        threshold (int):
            Time threshold in minutes for cumulative accessibility (default: 60).
        n (int or list[int]):
            Threshold(s) for number of opportunities in "cost_to_closest" method.
        group_by (list[str] or None):
            Additional columns in `travel_matrix` to segment accessibility results
            (e.g., by 'scenario', 'time_period').
        active (bool):
            If True, compute accessibility from origins (active); otherwise, to
            destinations (passive).
        fill_missing_ids (bool):
            For "cost_to_closest", whether to include all combinations of ID and
            `n` even if some are unreachable. Fills with `inf` if needed.

    Returns:
        pd.DataFrame or gpd.GeoDataFrame:
            Accessibility results. For "cumulative", returns GeoDataFrame with
            columns 'access_raw' and 'access_norm'. For "cost_to_closest", returns
            DataFrame with columns: ['id', ..., 'n', 'min_cost'].
    """   
    if group_by is None:
        group_by = []

    data = merge_opportunities(travel_matrix, land_uses, opportunity, active)

    id_col = "from_id" if active else "to_id"
    group_cols = [id_col] + group_by

    if method == "cumulative":
        return run_cumulative_method(
            data, land_uses, opportunity, threshold, id_col, group_cols
        )

    if method == "cost_to_closest":
        if isinstance(n, int):
            n = [n]

        scores = run_cost_to_closest_method(
            data, travel_matrix, opportunity, travel_cost_col,
            n, id_col, group_by, fill_missing_ids
        )

        return attach_geometries_to_cost_result(scores, land_uses)
    

def attach_geometries_to_cost_result(scores, land_uses):
    return land_uses[["hex_id", "geometry"]].merge(
        scores,
        left_on="hex_id",
        right_on="id",
        how="right"
    ).drop(columns=["id"])


In [ ]:
access = compute_accessibility(
    travel_matrix=ttime,
    land_uses=pois_by_hex.reset_index(),
    opportunity=tipo_oportunidade,
    method="cost_to_closest",
    threshold=tempo_maximo_viagem,
    n=[1],
    group_by=None,
    active=True,
    fill_missing_ids=True
)

# Visualização dos resultados

In [ ]:
def _setup_figure_ax(gdf: gpd.GeoDataFrame, figsize_width: float = 10.0):
    """Configura a figura e os eixos, mantendo a proporção dos dados."""
    minx, miny, maxx, maxy = gdf.total_bounds
    aspect_ratio = (
        (maxy - miny) / (maxx - minx) if (maxx - minx) != 0 else 1
    )
    figsize_height = figsize_width * aspect_ratio
    fig, ax = plt.subplots(figsize=(figsize_width, figsize_height))
    return fig, ax


def _finalize_plot(
    fig: plt.Figure,
    ax: plt.Axes,
    title: str,
    output_path: str | None,
    dpi: int,
):
    """Ajusta o título, remove eixos e salva ou mostra o plot."""
    ax.set_title(title, fontsize=16, pad=10)
    ax.axis("off")
    fig.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=dpi, bbox_inches="tight")
        print(f"Mapa salvo em: {output_path}")
    
    plt.show()
    plt.close(fig)


# Função principal

def plot_access_map(
    gdf: gpd.GeoDataFrame,
    value_col: str,
    title: str = "Mapa de Calor de Acessibilidade",
    cmap: str = "viridis",
    scheme: str = None,
    k: int = None,
    basemap_provider=ctx.providers.CartoDB.PositronNoLabels,
    scalebar_units: str = "m",
    north_arrow_location: str = "upper right",
    output_path: str | None = None,
    dpi: int = 600,
    figsize_width: float = 10.0,
    missing_values_color: str = "white",
):
    """
    Gera um mapa coroplético de contagens de varejo em grades H3.

    Parâmetros:
        gdf (gpd.GeoDataFrame): GeoDataFrame com geometrias H3 e dados.
        value_col (str): Coluna com os valores para o mapa coroplético.
        title (str): Título do mapa.
        cmap (str): Colormap Matplotlib.
        scheme (str): Esquema de classificação Mapclassify
                      (ex: 'Quantiles', 'FisherJenks').
        k (int): Número de classes para o esquema.
        basemap_provider: Provedor de mapa base do Contextily.
        scalebar_units (str): Unidades da barra de escala ('m' ou 'km').
        north_arrow_location (str): Localização da seta norte.
        output_path (str | None): Caminho para salvar a imagem.
                                  Se None, exibe o mapa.
        dpi (int): Resolução da imagem salva.
        figsize_width (float): Largura da figura em polegadas.
                               A altura é ajustada pela proporção.
        missing_values_color (str): Cor para geometrias com valores
                                    ausentes (NaN).
        legend_title (str | None): Título para a legenda. Padrão é o
                                   nome da value_col.
    """
    fig, ax = _setup_figure_ax(gdf, figsize_width)

    gdf_plot = gdf.copy()
    if gdf_plot[value_col].isnull().any():
        print(
            f"Aviso: {gdf_plot[value_col].isnull().sum()} valores nulos "
            f"encontrados em '{value_col}'. Serão representados por "
            f"'{missing_values_color}'."
        )

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    gdf_plot.plot(
        column=value_col,
        ax=ax,
        legend=True,
        cmap=cmap,
        legend_kwds={
            'cax': cax, # Pass the colorbar axes
            'label': 'Minutos' # Use the value column name as the legend title
        },
        scheme=scheme,
        linewidth=0,
        antialiased=True,
        alpha=.7,
        k=k,
        missing_kwds={
            "color": missing_values_color,
            'label': 'Nulos'
        },
    )
    cax.tick_params(axis='y', labelsize=12)
    cax.yaxis.label.set_fontsize(14)
            


    ctx.add_basemap(
        ax,
        crs=gdf.crs.to_string(),
        source=basemap_provider,
        attribution_size=5,
    )
    north_arrow(
        ax,
        location=north_arrow_location,
        rotation={"crs": gdf_plot.crs, "reference": "center"}
    )

    if gdf.crs.is_projected:
        scalebar = ScaleBar(
            dx=1.0,
            units=scalebar_units,
            location="lower right",
            scale_loc="bottom",
            box_alpha=0.85,
            frameon=False,
            font_properties={"size": "medium"},
        )
        ax.add_artist(scalebar)
    else:
        print(
            "Barra de escala não adicionada: o CRS do GeoDataFrame não "
            "é projetado."
        )

    _finalize_plot(fig, ax, title, output_path, dpi)
    

In [ ]:
outpath = Path(
    f"../outputs/mapas/acesso/{tipo_oportunidade}-complexidade_{nivel_complexidade}-hospital_{apenas_hospital}-30min.png"
    )

plot_access_map(
    access,#[access.min_cost <= tempo_maximo_viagem],
    value_col="min_cost",
    title="Tempo até a Oportunidade Mais Próxima",
    cmap="plasma_r",
    basemap_provider=ctx.providers.CartoDB.VoyagerNoLabels,  # Mapa base
    missing_values_color='lightgrey',
    #output_path=outpath,
)
plt.show()

# População v. Tempo

In [ ]:
from typing import List
from tobler.area_weighted import area_interpolate

def import_census_2022(ibge_ids: List[int|str], gcloud_id: str) -> gpd.GeoDataFrame:
    query = (
        f"SELECT id_setor_censitario, pessoas, geometria, {', '.join(f'V{i:0>5}' for i in range(644, 657))} "
        "FROM basedosdados.br_ibge_censo_2022.setor_censitario "
        f"WHERE id_municipio IN ({_sql_style_list(ibge_ids)}) "
    )

    data = bd.read_sql(query, billing_project_id=gcloud_id)

    return (
        gpd
        .GeoDataFrame(
            data,
            geometry=gpd.GeoSeries.from_wkt(data["geometria"]),
            crs=4326,
        )
        .drop(columns="geometria")
        .to_crs(31983)
        .rename(
        columns={'pessoas': 'habitantes'}
        )
    )


def _sql_style_list(sequence):
    """
    Converts a list of values into a SQL-style list of single-quoted strings.
    """
    return ", ".join(f"'{item}'" for item in sequence)


In [ ]:
tracts = import_census_2022(ibge_ids, gcloud_id)

In [ ]:

pop_by_age = area_interpolate(
    source_df=tracts.fillna(0),
    target_df=h3fy(
        tracts.assign(geometry=tracts.geometry.buffer(15)),
        resolution=9,
        ),
    extensive_variables=['habitantes'] + [f'V{i:0>5}' for i in range(644, 657)],
    allocate_total=True
)

pop_by_age["age_0_to_14"] = (
    pop_by_age
    .habitantes
    .sub(
        pop_by_age[[f'V{i:0>5}' for i in range(644, 657)]].sum(axis=1),
        axis='index',
        fill_value=0,
        )
    )

pop_by_age["age_15_to_19"] = (
    pop_by_age[['V00644']].sum(axis=1)
    )

pop_by_age["age_60_plus"] = (
    pop_by_age[[f'V{i:0>5}' for i in range(653, 657)]].sum(axis=1)
    )


In [ ]:
pop_by_age.explore('habitantes', cmap='magma', legend=True, scheme='headtailbreaks')

In [ ]:
def bin_trips(ttimes, pop, pop_group='age_0_to_14'):
    bins = np.arange(0, ttimes.min_cost.max() + 1, 1)
    labels = [b for b in bins[1:]]

    ttimes['travel_time_bins'] = pd.cut(
        ttimes['min_cost'].fillna(121),
        bins=bins,
        labels=labels,
        include_lowest=True,
        )

    return (
        pop[[pop_group, 'geometry']]
        .merge(
            ttimes.set_index('hex_id'),
            left_index=True,
            right_index=True,
            )
        .groupby(
            ['travel_time_bins'],
            #as_index=False,
            observed=True,
            )
        .agg({pop_group: 'sum'})
        .astype({
            pop_group: 'int',
            })
        )

In [ ]:
plt.style.use('ggplot')
age_group = 'age_0_to_14'

df = bin_trips(access, pop_by_age, pop_group=age_group)
sns.ecdfplot(x=df.index,weights=df[age_group])
sns.despine()
plt.xlabel("Tempo de Viagem (minutos)")
plt.ylabel("Proporção(%)")
ax = plt.gca()
labels = [f"{tick*100:.0f}" for tick in ax.get_yticks()]
ax.set_yticklabels(labels)
plt.show()



In [ ]:
inpath = tabelas_atributos = (
    '../database/1. Socioeconômicos/sociodemografia_2010.parquet'
)
census_data = gpd.read_parquet(inpath)

In [ ]:
foo = (
    census_data
    #.merge(pop_by_age[['habitantes']], left_index=True, right_index=True)
    .merge(access, left_index=True, right_on='hex_id')
    )

bins = [0, 15, 30, 60, np.inf]
foo['travel_time_bins'], x = pd.cut(
    foo['min_cost'].fillna(np.inf),
    bins=bins,
    labels=bins[:-1],
    include_lowest=True,
    retbins=True
)


In [ ]:
pd.cut(
    foo['min_cost'],
    bins=bins,
    labels=bins[:-1],
    include_lowest=True,
    retbins=True
)



In [ ]:
def reindex_df(df, weight_col):
    df = df.reindex(df.index.repeat(df[weight_col]))
    df.reset_index(drop=True, inplace=True)
    return(df)


def weighted_boxplot(df, weight_col):
    sns.boxplot(x='travel_time_bins', 
                y='rendimento_medio', 
                data=reindex_df(df, weight_col=weight_col),)
    plt.show()

weighted_boxplot(foo, 'habitantes')

In [ ]:
x

In [ ]:
(
    census_data
    .merge(pop_by_age[['age_0_to_14']], left_index=True, right_index=True)
    .merge(access, left_index=True, right_on='hex_id')
    )

In [ ]:
foo.travel_time_bins

In [ ]:
def classify_income_deciles(
        df,
        income_col='rendimento_medio',
        weight_col='habitantes',
        ):
    d = DescrStatsW(df[income_col], weights=df[weight_col])
    # Compute the quantile breakpoints for deciles
    quantiles = [d.quantile(q) for q in np.linspace(0.1, 1.0, 10)]
    bins = [df[income_col].min() - 1] + [float(q) for q in quantiles]
    
    # Assign decile labels 1–10
    return pd.cut(df[income_col], bins=bins, labels=range(1, 11), include_lowest=True)

In [ ]:
classify_income_deciles(census_data)

# Legacy

In [ ]:
from shapely.geometry import Point
from pyproj import Transformer
# Original coordinates are in (lon, lat) WGS84 (EPSG:4326)

transformer = Transformer.from_crs("EPSG:4326", "EPSG:31983", always_xy=True)

d = {
    k: Point(*transformer.transform(pt.x, pt.y))
    for k, pt in {
        '5042488': Point(-43.10111846368994, -22.93496303213362),
        '0113891': Point(-42.97796431846613, -22.82297090020438),
        '0012521': Point(-43.07872347303554, -22.880157833214476),
        '3784916': Point(-42.91998438196854, -22.771568234552003), 
        '0012599': Point(-43.078428218305724, -22.881199876918767),
        '2297590': Point(-43.01194919623843, -22.81838059154707),
        '9101039': Point(-42.95904980987253, -22.84514982733307),
    }.items()
}

for k, pt in d.items():
    pois.at[k, 'geometry'] = pt

pois = pois.drop(index='9101039')